Этот нотбук выполняет обучение и оптимизацию бинарного классификатора для разделения постов на вакансии и резюме:
1. Загрузка размеченного вручную датасета `manual_labeling_sample.csv`.
2. Лингвистическая предобработка текста.
3. Разделение данных на обучающую и тестовую выборки (сбалансированное разбиение).
4. Настройка конвейера (Pipeline): TF-IDF векторизация + логистическая регрессия.
5. Оптимизация гиперпараметров через `GridSearchCV` с кросс-валидацией.
6. Вывод детальных метрик качества (`classification_report` и `confusion_matrix`).
7. Анализ наиболее значимых текстовых признаков (весов классификатора).
8. Экспорт готового пайплайна в `models/best_binary_classifier.joblib`.

In [ ]:
import os
import re
import joblib
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from typing import List, Dict, Tuple, Any

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore")

# Настройки визуализации
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["font.size"] = 12

In [ ]:
# Безопасная фоновая загрузка стоп-слов NLTK
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords", quiet=True)

RU_STOPWORDS = set(stopwords.words("russian"))
EN_STOPWORDS = set(stopwords.words("english"))
ALL_STOPWORDS = RU_STOPWORDS.union(EN_STOPWORDS)

def clean_raw_text(text: str) -> str:
    """
    Выполняет глубокую очистку текста для построения признаков:
    - Удаляет веб-ссылки, Telegram-ссылки, телефоны и юзернеймы.
    - Переводит все знаки конца предложения (!, ?) в точки.
    - Удаляет всю пунктуацию, кроме букв, цифр и точек.
    - Фильтрует русские и английские стоп-слова.
    - Сохраняет одиночные точки (.) для предотвращения склеивания слов на границах предложений.
    """
    text_str = str(text).lower()

    # 1. Удаление веб-ссылок, ссылок на Telegram-каналы/сообщения и телефонных номеров
    text_str = re.sub(r"https?://\S+|www\.\S+", " ", text_str)
    text_str = re.sub(r"\bt\.me/\S+|@\S+", " ", text_str)
    text_str = re.sub(r"(\+?\d[\d\s\-\(\)]{7,}\d)", " ", text_str)

    # 2. Превращение знаков конца предложения в единую точку для сохранения границ
    text_str = re.sub(r"[!?\n\r\t]+", ". ", text_str)

    # 3. Удаление всех спецсимволов и пунктуации, кроме кириллицы, латиницы, цифр и точек
    text_str = re.sub(r"[^a-zа-яё0-9.\s]", " ", text_str)

    # 4. Пословная фильтрация стоп-слов с сохранением семантических точек
    raw_tokens = text_str.split()
    cleaned_tokens: List[str] = []

    for token in raw_tokens:
        # Обрабатываем токен, если он является самостоятельной точкой
        if token == ".":
            if not cleaned_tokens or cleaned_tokens[-1] != ".":
                cleaned_tokens.append(".")
            continue

        # Обрабатываем слова, которые заканчиваются на точку (конец предложения)
        if token.endswith("."):
            clean_word = token[:-1].strip()
            if clean_word and clean_word not in ALL_STOPWORDS and not clean_word.isdigit():
                cleaned_tokens.append(clean_word)
            if not cleaned_tokens or cleaned_tokens[-1] != ".":
                cleaned_tokens.append(".")
            continue

        # Обрабатываем стандартные слова
        if token not in ALL_STOPWORDS and not token.isdigit():
            cleaned_tokens.append(token)

    # Склеиваем слова обратно в текст
    result_text = " ".join(cleaned_tokens)
    
    # Лингвистическое форматирование: убираем пробел перед точкой ("слово ." -> "слово.")
    result_text = re.sub(r"\s+\.", ".", result_text)
    
    # Схлопываем множественные точки в одну ("слово..." -> "слово.")
    result_text = re.sub(r"\.+", ".", result_text)
    
    # Удаляем лишние пробелы
    result_text = re.sub(r"\s+", " ", result_text).strip()

    return result_text

In [ ]:
DATA_DIR = "data"
MODELS_DIR = "models"
os.makedirs(MODELS_DIR, exist_ok=True)

LABELED_DATA_PATH = os.path.join(DATA_DIR, "manual_labeling_sample.csv")

if not os.path.exists(LABELED_DATA_PATH):
    raise FileNotFoundError(
        f"Файл ручной разметки {LABELED_DATA_PATH} не найден. "
        "Пожалуйста, убедитесь, что вы положили файл manual_labeling_sample.csv в папку data."
    )

# Загрузка датасета
df_labeled_raw = pd.read_csv(LABELED_DATA_PATH)

df_labeled = pd.DataFrame()
df_labeled["original_text"] = df_labeled_raw["text"].fillna("").astype(str)
df_labeled["preliminary_label"] = df_labeled_raw["preliminary_label"].str.strip().str.lower()

# Отбираем только размеченные как "vacancy" или "resume" классы
df_train_full = df_labeled[df_labeled["preliminary_label"].isin(["vacancy", "resume"])].copy()

print(f"Размер отфильтрованного размеченного датасета: {len(df_train_full)}")
print("Распределение классов до очистки:")
print(df_train_full["preliminary_label"].value_counts())

# Очистка текстов
print("\nЗапуск очистки текстов...")
df_train_full["cleaned_text"] = df_train_full["original_text"].apply(clean_raw_text)

# Исключаем строки, которые стали пустыми после очистки
df_train_full = df_train_full[df_train_full["cleaned_text"].str.strip() != ""].copy()
print(f"Размер датасета после удаления пустых очищенных строк: {len(df_train_full)}")

In [ ]:
# Разделение на признаки и целевые метки
X = df_train_full["cleaned_text"].values

# Вакансии кодируем как 1, Резюме как -1
y = np.where(df_train_full["preliminary_label"] == "vacancy", 1, -1)

# Разделение на обучение и тест (80% / 20%) со стратификацией классов
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Размер обучающей выборки: {len(X_train)}")
print(f"Размер тестовой выборки:  {len(X_test)}")

In [ ]:
# Настройка TF-IDF
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    token_pattern=r"(?u)\b[а-яёa-z][а-яёa-z0-9_-]{2,}\b"  # Защита от одиночных символов
)

# Определение базовой модели классификации
classifier = LogisticRegression(
    class_weight="balanced",
    solver="saga",
    random_state=42,
    max_iter=1000
)

# Объединение в Pipeline
pipeline = Pipeline([
    ("tfidf", vectorizer),
    ("clf", classifier)
])

# Пространство поиска гиперпараметров
param_grid = {
    "clf__penalty": ["l2", "l1"],
    "clf__C": [0.1, 0.5, 1.0, 5.0, 10.0]
}

# Настройка кросс-валидации
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Оптимизация параметров
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

print("Запуск оптимизации гиперпараметров классификатора...")
grid_search.fit(X_train, y_train)

print("\nРезультаты оптимизации:")
print(f"  Лучшие параметры: {grid_search.best_params_}")
print(f"  Наилучший Macro F1 на валидации: {grid_search.best_score_:.4f}")

In [ ]:
# Получение лучшего классификатора по результатам кросс-валидации
best_pipeline = grid_search.best_estimator_

# Предсказание классов на отложенной выборке (20%)
y_pred = best_pipeline.predict(X_test)

# Вывод метрик классификации
print("=" * 60)
print("ОТЧЕТ О КАЧЕСТВЕ НА ТЕСТОВОЙ ВЫБОРКЕ (TEST SET)")
print("=" * 60)

target_names = ["Resume (-1)", "Vacancy (1)"]
print(classification_report(y_test, y_pred, target_names=target_names))

# Построение Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=target_names,
    yticklabels=target_names,
    cbar=False
)
plt.title("Confusion Matrix (Test Set)")
plt.ylabel("Фактический класс")
plt.xlabel("Предсказанный класс")
plt.tight_layout()
plt.show()

In [ ]:
def extract_feature_importance(pipeline: Pipeline, top_n: int = 15):
    """
    Извлекает и отображает веса наиболее значимых слов для классов Вакансия и Резюме.
    """
    vectorizer_step = pipeline.named_steps["tfidf"]
    classifier_step = pipeline.named_steps["clf"]

    feature_names = np.array(vectorizer_step.get_feature_names_out())
    weights = classifier_step.coef_[0]

    sorted_indices = weights.argsort()

    # Топ положительных весов -> Вакансии
    print("=" * 65)
    print(f"ТОП {top_n} СИГНАЛОВ ДЛЯ ВАКАНСИЙ (Положительные веса)")
    print("=" * 65)
    vacancy_features = []
    for idx in sorted_indices[::-1]:
        w = float(weights[idx])
        if w > 0.0001:  # отсекаем нулевые признаки
            vacancy_features.append((feature_names[idx], w))
        if len(vacancy_features) >= top_n:
            break

    for rank, (word, weight) in enumerate(vacancy_features, 1):
        print(f"  {rank:<2} {word:<30} | Вес: +{weight:.4f}")

    # Топ отрицательных весов -> Резюме
    print("\n" + "=" * 65)
    print(f"ТОП {top_n} СИГНАЛОВ ДЛЯ РЕЗЮМЕ (Отрицательные веса)")
    print("=" * 65)
    resume_features = []
    for idx in sorted_indices:
        w = float(weights[idx])
        if w < -0.0001:  # отсекаем нулевые признаки
            resume_features.append((feature_names[idx], w))
        if len(resume_features) >= top_n:
            break

    for rank, (word, weight) in enumerate(resume_features, 1):
        print(f"  {rank:<2} {word:<30} | Вес: {weight:.4f}")

extract_feature_importance(best_pipeline, top_n=15)

In [ ]:
# Экспорт обученного пайплайна
model_output_path = os.path.join(MODELS_DIR, "best_binary_classifier.joblib")
joblib.dump(best_pipeline, model_output_path)

print("=" * 60)
print(f"Оптимальный пайплайн сохранен в: {model_output_path}")
print("=" * 60)